# Extracción

Importación de librerías y lectura de los datos crudos (players, teams, stadiums) desde Google Sheets.

In [1]:
# Librerías usadas en el notebook:
# - pandas: manipulación de datos en DataFrames (leer CSV, unir/transformar tablas, etc.)
# - os: utilidades del sistema operativo (aquí se usa para crear la carpeta "bd_actual" al guardar los CSV finales)
import pandas as pd
import os

In [2]:
SHEET_ID = "1pDriJ9b7K2Vnp_MP5UC2xmSkhpoZ3xN0K-r7hasD0SE" #id de la hoja se obtiene de la URL
GID_PLAYERS = "1757951370" # Hoja de players
GID_TEAMS = "392316125" # Hoja de tems
GID_STADIUMS = "1508925939"

# Función que permite leer cada una de las hojas del link
# Recibe como parametro gun GID
# Retorna la lectura de la hoja utilizando el link
def read_tab(gid):
    url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={gid}"
    return pd.read_csv(url)

In [3]:
# Se lee players, stadiums, teams, utilizando la función definida arriba
raw_players = read_tab(GID_PLAYERS)
raw_teams = read_tab(GID_TEAMS)
raw_stadiums = read_tab(GID_STADIUMS)

raw_players.shape # Nos dice cuantas filas y columnas hay


(576, 8)

# Transformación

Creación de las tablas normalizadas y carga de teams, stadiums y players a partir de los datos crudos.

In [4]:
# ============================================================
# 3. Creación de las tablas normalizadas (vacías)
# ============================================================
# Estas son las tablas "limpias" del modelo final, todavía vacías.
# Se van a ir llenando poco a poco con funciones get_or_create_* que
# evitan duplicados (por ejemplo, si un equipo ya existe no se vuelve
# a insertar, solo se reutiliza su id).
#
# - world_cups: catálogo de mundiales (uno por edición, ej. Qatar 2022)
# - players / teams / stadiums: catálogos únicos (sin duplicados,
#   reutilizables entre distintos mundiales)
# - player_world_cup / team_world_cup / stadium_world_cup: tablas
#   puente que relacionan cada catálogo con un mundial específico
#   (por eso llevan world_cup_id)
world_cups = pd.DataFrame(columns=["id", "name", "year","host_country"])
players = pd.DataFrame(columns=["id", "name", "birthday", "debut"])
teams = pd.DataFrame(columns=["id", "name", "code"])
stadiums = pd.DataFrame(columns=["id", "name", "capacity", "latitude", "longitude"])
player_world_cup = pd.DataFrame(columns=["id", "player_id", "team_id","world_cup_id", "weight", "height","position"])
team_world_cup = pd.DataFrame(columns=["id", "team_id","world_cup_id","group"])
stadium_world_cup = pd.DataFrame(columns=["id","stadium_id","world_cup_id"])


world_cups.shape


(0, 4)

In [5]:
# ============================================================
# 4. Alta del mundial actual (upsert de world_cups)
# ============================================================
# "Upsert" = si el mundial ya existe en world_cups, reutiliza su id;
# si no existe, lo crea con el siguiente id disponible. Así, si esta
# celda se vuelve a correr, no se duplica el registro de "Qatar 2022".
WORLD_CUP_NAME, WORLD_CUP_YEAR, HOST = "Qatar 2022", 2022, "Qatar"
match = world_cups[world_cups["name"] == WORLD_CUP_NAME]

if len(match) > 0:
    world_cup_id = int(match.iloc[0]["id"])
else:
    world_cup_id = int(world_cups["id"].max() + 1 if len(world_cups) > 0 else 1)

fila_nueva = pd.DataFrame([{
    "id": world_cup_id,
    "name": WORLD_CUP_NAME,
    "year": WORLD_CUP_YEAR,
    "host_country": WORLD_CUP_YEAR  # NOTA: guarda el año, no HOST ("Qatar"); revisar si es un typo.
}])
world_cups = pd.concat([world_cups, fila_nueva], ignore_index=True)

# world_cup_id queda disponible para todas las celdas siguientes: es
# la llave foránea que conecta teams/stadiums/players con este mundial.
print(f"world_cup_id para {WORLD_CUP_NAME}: {world_cup_id}" )



world_cup_id para Qatar 2022: 1


In [6]:
# ============================================================
# Chequeo rápido: comparar columnas de la tabla normalizada "teams"
# contra las columnas crudas "raw_teams". Es solo un print de
# diagnóstico para confirmar cómo se van a mapear los campos antes
# de escribir get_or_create_team_id.
# ============================================================
print("teams:", teams.columns.tolist())
print("raw_teams:", raw_teams.columns.tolist())

teams: ['id', 'name', 'code']
raw_teams: ['id', 'name', 'abbreviation', 'championships', 'group']


## Cargar Teams y team_world_cup

In [7]:
# ============================================================
# get_or_create_team_id: mismo patrón "upsert" que el mundial (paso 4),
# pero para equipos. Busca el equipo por su "code" (abreviación, ej.
# "ARG"); si ya existe reutiliza su id GLOBAL, si no lo crea. Esto
# evita tener el mismo equipo duplicado en "teams" si aparece en más
# de un mundial.
# ============================================================
def get_or_create_team_id(teams_df, name, code):
    match = teams_df[teams_df["code"] == code]
    if len(match) > 0:
        return teams_df, int(match.iloc[0]["id"])

    new_id = int(teams_df["id"].max() + 1 if len(teams_df) > 0 else 1)
    new_row = pd.DataFrame([{
        "id": new_id,
        "name": name,
        "code": code
    }])
    teams_df = pd.concat([teams_df, new_row], ignore_index=True)
    return teams_df, new_id

# team_id_map: traduce el "id" LOCAL de raw_teams (el id de esa hoja
# de Google Sheets) al "id" GLOBAL en la tabla normalizada teams. Se
# necesita porque distintos mundiales pueden traer ids distintos para
# el mismo equipo, y más adelante (paso 6, players) hay que saber a
# qué equipo GLOBAL pertenece cada jugador.
team_id_map = {}
for _, fila in raw_teams.iterrows():
    # 1) Upsert del equipo en la tabla global "teams"
    teams, global_team_id = get_or_create_team_id(teams, fila["name"], fila["abbreviation"])
    team_id_map[fila["id"]] = global_team_id

    # 2) Registrar la participación de este equipo en ESTE mundial
    #    (tabla puente team_world_cup), con su grupo (A, B, C...)
    new_id = int(team_world_cup["id"].max()) + 1 if len(team_world_cup) > 0 else 1

    nueva_participacion = pd.DataFrame([{
        "id": new_id, "team_id": global_team_id, "world_cup_id": world_cup_id, "group": fila["group"],
    }])
    team_world_cup = pd.concat([team_world_cup, nueva_participacion], ignore_index=True)

    print("\nMapeo team_id local -> global:", team_id_map)



Mapeo team_id local -> global: {1: 1}

Mapeo team_id local -> global: {1: 1, 2: 2}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12}

Mapeo team_id local -> global: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13}

Mape

In [8]:
# ============================================================
# 5. Cargar stadiums + stadium_world_cup
# ============================================================
# Exactamente el mismo patrón "upsert" que teams (busca por "name" en
# vez de "code" porque los estadios no traen un código corto).
# stadium_world_cup es la tabla puente: qué estadios se usaron en
# este mundial.

def get_or_create_stadium_id(stadiums_df, name, capacity, lat, lon):
    match = stadiums_df[stadiums_df["name"] == name]
    if len(match) > 0:
        return stadiums_df, int(match.iloc[0]["id"])

    new_id = int(stadiums_df["id"].max()) + 1 if len(stadiums_df) > 0 else 1
    new_row = pd.DataFrame([{
        "id": new_id, "name": name, "capacity": capacity, "latitude": lat, "longitude": lon,
    }])
    stadiums_df = pd.concat([stadiums_df, new_row], ignore_index=True)
    return stadiums_df, new_id


for _, fila in raw_stadiums.iterrows():
    # 1) Upsert del estadio en la tabla global "stadiums"
    stadiums, global_stadium_id = get_or_create_stadium_id(
        stadiums, fila["name"], fila["capacity"], fila["latitude"], fila["longitude"]
    )
    # 2) Registrar que este estadio se usó en ESTE mundial
    new_id = int(stadium_world_cup["id"].max()) + 1 if len(stadium_world_cup) > 0 else 1
    nueva_participacion = pd.DataFrame([{
        "id": new_id, "stadium_id": global_stadium_id, "world_cup_id": world_cup_id,
    }])
    stadium_world_cup = pd.concat([stadium_world_cup, nueva_participacion], ignore_index=True)


In [9]:
# ============================================================
# 6. Cargar players + player_world_cup (usa el mapeo de teams)
# ============================================================
# Mismo patrón "upsert" que teams y stadiums, pero para jugadores. La
# diferencia es que no hay un código corto como "ARG": para saber si
# un jugador ya existe hay que comparar su nombre, así que primero se
# normaliza (normalize_name) para que mayúsculas/espacios no generen
# duplicados falsos.

def normalize_name(name):
    """
    Deja un nombre en formato comparable, paso a paso:
    1) .strip()  -> quita espacios sobrantes al inicio/final
    2) .upper()  -> convierte todo a mayúsculas
    3) .split()  -> separa el texto en una LISTA de palabras
                    (esto además colapsa espacios dobles entre palabras)
    4) " ".join(lista) -> vuelve a unir esa lista en un solo string,
                           con exactamente un espacio entre palabras
    Ejemplo: "  saad   al sheeb " -> "SAAD AL SHEEB"
    """
    return " ".join(name.strip().upper().split())


def get_or_create_player_id(players_df, name, birthday, debut):
    """
    Busca si el jugador ya existe, comparando nombres NORMALIZADOS (no
    el texto crudo, para no fallar por mayúsculas o espacios distintos).
    Si existe, regresa su id. Si no, lo agrega como jugador nuevo.
    """
    norm_name = normalize_name(name)  # normaliza el nombre que estamos buscando

    if len(players_df) > 0:
        # .assign() crea una COPIA de players_df con una columna extra
        # "_norm" (el nombre de cada fila ya normalizado), sin tocar
        # la tabla real. .apply() corre normalize_name en cada fila.
        existentes = players_df.assign(_norm=players_df["name"].apply(normalize_name))
        # Compara esa columna temporal contra el nombre que buscamos.
        match = existentes[existentes["_norm"] == norm_name]
    else:
        match = players_df  # tabla vacía: no hay nada que comparar todavía

    if len(match) > 0:
        return players_df, int(match.iloc[0]["id"])  # ya existía

    # No existía: crea el jugador nuevo con el siguiente id disponible.
    new_id = int(players_df["id"].max()) + 1 if len(players_df) > 0 else 1
    new_row = pd.DataFrame([{
        "id": new_id,
        "name": name,
        "birthday": pd.to_datetime(birthday),  # convierte el texto de fecha a un dato tipo fecha real
        "debut": pd.to_datetime(debut),
    }])
    players_df = pd.concat([players_df, new_row], ignore_index=True)
    return players_df, new_id


for _, fila in raw_players.iterrows():
    # 1) Upsert del jugador en la tabla global "players"
    players, player_id = get_or_create_player_id(players, fila["name"], fila["birthday"], fila["debut"])

    # El team_id que trae la fila cruda es LOCAL a este Sheet. Lo
    # buscamos en team_id_map (construido en el paso 4) para traducirlo
    # al id GLOBAL correcto de la tabla teams.
    global_team_id = team_id_map[fila["team_id"]]

    # 2) Registrar la participación de este jugador en ESTE mundial,
    #    con el equipo, peso, altura y posición que trae esa edición.
    new_id = int(player_world_cup["id"].max()) + 1 if len(player_world_cup) > 0 else 1
    nueva_participacion = pd.DataFrame([{
        "id": new_id, "player_id": player_id, "team_id": global_team_id, "world_cup_id": world_cup_id,
        "height": fila["height"], "weight": fila["weight"], "position": fila["position"],
    }])
    player_world_cup = pd.concat([player_world_cup, nueva_participacion], ignore_index=True)

# ============================================================
# 7. Validaciones
# ============================================================
# Chequeos de calidad sobre las tablas ya construidas, antes de darlas
# por buenas: ¿hay nombres de jugadores duplicados?, ¿hay códigos de
# equipo duplicados?, ¿todas las referencias en player_world_cup
# apuntan a un jugador que sí existe en players? Si todo sale "OK",
# las tablas están listas para pasar a la etapa de Carga.

print("\n--- Validación: nombres duplicados en players ---")
# players["name"].apply(normalize_name) -> una columna con todos los
# nombres ya normalizados.
# .duplicated(keep=False) marca con True TODAS las filas que compartan
# un valor repetido (no solo la 2a ocurrencia en adelante, como haría
# .duplicated() por default).
nombres_normalizados = players["name"].apply(normalize_name)
dup_players = players[nombres_normalizados.duplicated(keep=False)]
print("OK, sin duplicados" if len(dup_players) == 0 else dup_players)

print("\n--- Validación: códigos duplicados en teams ---")
dup_teams = teams[teams["code"].duplicated(keep=False)]
print("OK, sin duplicados" if len(dup_teams) == 0 else dup_teams)

print("\n--- Validación: integridad referencial en player_world_cup ---")
# player_world_cup["player_id"].isin(players["id"]) -> True/False por
# cada fila, según si ese player_id SÍ aparece en la tabla players.
# El ~ de enfrente invierte eso: nos quedamos con las filas donde el
# player_id NO existe en players -> "referencias huérfanas" (un bug).
huerfanos = player_world_cup[~player_world_cup["player_id"].isin(players["id"])]
print("OK, todos los player_id existen en players" if len(huerfanos) == 0 else huerfanos)

# ============================================================
# 8. Resultado final
# ============================================================
# Imprime cada una de las 7 tablas normalizadas ya construidas, solo
# para revisarlas visualmente antes de pasar a la etapa de Carga
# (donde se escriben a disco).

# Un diccionario donde cada llave es el nombre de la tabla (para
# imprimirlo) y cada valor es el DataFrame correspondiente.
tablas = {
    "world_cups": world_cups, "teams": teams, "stadiums": stadiums, "players": players,
    "team_world_cup": team_world_cup, "stadium_world_cup": stadium_world_cup,
    "player_world_cup": player_world_cup,
}

# .items() recorre el diccionario dándote pares (llave, valor) en cada
# vuelta del for: aquí, (nombre_de_la_tabla, el_dataframe_en_sí).
for nombre, df in tablas.items():
    print(f"\n--- {nombre} ---")
    print(df)



--- Validación: nombres duplicados en players ---
OK, sin duplicados

--- Validación: códigos duplicados en teams ---
OK, sin duplicados

--- Validación: integridad referencial en player_world_cup ---
OK, todos los player_id existen en players

--- world_cups ---
  id        name  year host_country
0  1  Qatar 2022  2022         2022

--- teams ---
    id            name code
0    1           QATAR  QAT
1    2         ECUADOR  ECU
2    3         SENEGAL  SEN
3    4     NETHERLANDS  NED
4    5         ENGLAND  ENG
5    6         IR IRAN  IRN
6    7             USA  USA
7    8          WALLES  WAL
8    9       ARGENTINA  ARG
9   10    SAUDI ARABIA  KSA
10  11          MEXICO  MEX
11  12          POLAND  POL
12  13          FRANCE  FRA
13  14       AUSTRALIA  AUS
14  15         DENMARK  DEN
15  16         TUNISIA  TUN
16  17           SPAIN  ESP
17  18      COSTA RICA  CRC
18  19         GERMANY  GER
19  20           JAPAN  JPN
20  21         BELGIUM  BEL
21  22          CANADA  CAN
22  

# Carga

Persistencia de las tablas normalizadas a disco (CSV). En un pipeline real este paso sería un `to_sql()` hacia PostgreSQL u otra base de datos.

In [10]:
# ============================================================
# 9. Persistir el estado actual (aquí en CSV; en tu DAG real sería
#    un to_sql() hacia PostgreSQL)
# ============================================================
# Recorremos el mismo diccionario "tablas" que armamos en el paso 8
# (nombre_de_la_tabla, el_dataframe correspondiente) y guardamos cada
# una como un archivo CSV dentro de la carpeta "bd_actual".
os.makedirs("bd_actual", exist_ok=True)

for nombre, df in tablas.items():
    # .to_csv(ruta, index=False) guarda el DataFrame como archivo CSV.
    # index=False evita que se guarde también el índice interno de
    # pandas (0,1,2...) como si fuera una columna más del archivo —
    # esa numeración es de pandas, no es dato real de la tabla.
    #
    # f"bd_actual/{nombre}.csv" arma el nombre del archivo usando el
    # nombre de la tabla, por ejemplo: bd_actual/players.csv,
    # bd_actual/teams.csv, etc.
    #
    # Por qué esto importa: sin este paso, todo lo que cargamos existe
    # solo en la memoria de este script — en cuanto termina de correr,
    # se pierde. Guardarlo es lo que permite que la SIGUIENTE carga
    # (por ejemplo, el mundial 2026) pueda arrancar leyendo este mismo
    # estado en vez de partir de tablas vacías otra vez.
    df.to_csv(f"bd_actual/{nombre}.csv", index=False)


## Insertar el modelo completo (7 tablas) en PostgreSQL

Borraste las tablas planas que había antes en el schema `worldcup`, así que ahora esta celda crea ahí mismo el modelo relacional **completo**: las 7 tablas normalizadas (`world_cups`, `teams`, `stadiums`, `players`, `team_world_cup`, `stadium_world_cup`, `player_world_cup`), con sus propias llaves primarias y foráneas — la misma integridad referencial que ya validamos "a mano" en el paso 7, pero ahora reforzada por la base de datos.

Notas:
- La primera vez que corras esta celda crea el schema y las tablas (`CREATE ... IF NOT EXISTS`). Las siguientes corridas reutilizan esas mismas tablas.
- Cada corrida hace `TRUNCATE` + `INSERT` de las 7 tablas para dejarlas sincronizadas con lo que se acaba de transformar en memoria (sin duplicar filas).
- Usuario y contraseña se leen de las variables de entorno `PGUSER`/`PGPASSWORD` (nunca quedan escritos en el notebook). Si tu Postgres local usa autenticación `trust` (como en una instalación por defecto de Homebrew), la contraseña no se valida y basta con exportar `PGUSER` con tu superusuario local.

In [ ]:
# ============================================================
# 10. Insertar el modelo completo (7 tablas) en PostgreSQL
# ============================================================
# SQLAlchemy arma el "engine" (la conexión) que pandas necesita para
# hablar con la base de datos. psycopg2-binary es el driver que usa
# por debajo para conectarse a Postgres.
from sqlalchemy import create_engine, text
import psycopg2

PG_HOST = "localhost"
PG_PORT = "5432"
PG_DB = "mundial2026"
PG_SCHEMA = "worldcup2026"
PG_USER = "postgres"
PG_PASSWORD = "Moncho2025#"

# Primero crear la base de datos si no existe (conectando a postgres)
try:
    # Conectar a la base de datos postgres por defecto
    conn = psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        database="postgres",
        user=PG_USER,
        password=PG_PASSWORD
    )
    conn.autocommit = True
    cursor = conn.cursor()
    
    # Verificar si la base de datos existe
    cursor.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PG_DB,))
    exists = cursor.fetchone()
    
    if not exists:
        cursor.execute(f"CREATE DATABASE {PG_DB}")
        print(f"Base de datos '{PG_DB}' creada exitosamente")
    else:
        print(f"Base de datos '{PG_DB}' ya existe")
    
    cursor.close()
    conn.close()
except Exception as e:
    print(f"Error al crear la base de datos: {e}")

# Ahora conectar a la base de datos mundial2026
engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}")

# ------------------------------------------------------------
# DDL: crea el schema y las 7 tablas SOLO si no existen todavía. Las
# FK dejan la integridad referencial garantizada por la base de datos
# (lo mismo que ya validamos "a mano" en el paso 7).
# ------------------------------------------------------------
DDL = f"""
CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA};

CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.world_cups (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    year INTEGER NOT NULL,
    host_country TEXT
);

CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.teams (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    code TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.stadiums (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    capacity INTEGER,
    latitude NUMERIC(9,6),
    longitude NUMERIC(9,6)
);

CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.players (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birthday DATE,
    debut DATE
);

CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.team_world_cup (
    id INTEGER PRIMARY KEY,
    team_id INTEGER NOT NULL REFERENCES {PG_SCHEMA}.teams(id),
    world_cup_id INTEGER NOT NULL REFERENCES {PG_SCHEMA}.world_cups(id),
    group_name TEXT
);

CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.stadium_world_cup (
    id INTEGER PRIMARY KEY,
    stadium_id INTEGER NOT NULL REFERENCES {PG_SCHEMA}.stadiums(id),
    world_cup_id INTEGER NOT NULL REFERENCES {PG_SCHEMA}.world_cups(id)
);

CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.player_world_cup (
    id INTEGER PRIMARY KEY,
    player_id INTEGER NOT NULL REFERENCES {PG_SCHEMA}.players(id),
    team_id INTEGER NOT NULL REFERENCES {PG_SCHEMA}.teams(id),
    world_cup_id INTEGER NOT NULL REFERENCES {PG_SCHEMA}.world_cups(id),
    weight INTEGER,
    height NUMERIC(3,2),
    position TEXT
);
"""

# Ejecutar el DDL
with engine.connect() as conn:
    conn.execute(text(DDL))
    conn.commit()

print("Schema y tablas creadas exitosamente en PostgreSQL")

# ------------------------------------------------------------
# Insertar datos: TRUNCATE + INSERT para cada tabla
# ------------------------------------------------------------
# Primero truncamos para evitar duplicados en corridas sucesivas
TRUNCATE = f"""
TRUNCATE TABLE {PG_SCHEMA}.player_world_cup,
                    {PG_SCHEMA}.team_world_cup,
                    {PG_SCHEMA}.stadium_world_cup,
                    {PG_SCHEMA}.players,
                    {PG_SCHEMA}.teams,
                    {PG_SCHEMA}.stadiums,
                    {PG_SCHEMA}.world_cups RESTART IDENTITY CASCADE;
"""

with engine.connect() as conn:
    conn.execute(text(TRUNCATE))
    conn.commit()

print("Tablas truncadas")

# Insertar datos en el orden correcto (respetando FK)
# Renombrar la columna "group" a "group_name" en team_world_cup
team_world_cup_renamed = team_world_cup.rename(columns={"group": "group_name"})

tablas = {
    "world_cups": world_cups, "teams": teams, "stadiums": stadiums, "players": players,
    "team_world_cup": team_world_cup_renamed, "stadium_world_cup": stadium_world_cup,
    "player_world_cup": player_world_cup,
}

tablas_orden = ["world_cups", "teams", "stadiums", "players", "team_world_cup", "stadium_world_cup", "player_world_cup"]

for nombre in tablas_orden:
    df = tablas[nombre]
    df.to_sql(nombre, engine, schema=PG_SCHEMA, if_exists="append", index=False)
    print(f"Insertados {len(df)} registros en {nombre}")

print("\n¡Carga completada en PostgreSQL!")

# ============================================================
# 11. Verificación de datos en PostgreSQL
# ============================================================
# Consultar las tablas para verificar que los datos se cargaron correctamente
with engine.connect() as conn:
    for nombre in tablas_orden:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {PG_SCHEMA}.{nombre}"))
        count = result.fetchone()[0]
        print(f"{nombre}: {count} registros")

In [ ]:
# Celda duplicada eliminada